# Final Modeling Notebook

This notebook is the execution layer for the final phase. It trains the two propensity models, evaluates them with a simple temporal holdout, parses the CRM rule file, and exports score and decision outputs.

Run `eda.ipynb` first so the shared snapshot artifacts are written to `artifacts/eda_training_artifacts.pkl`.

In [110]:
from pathlib import Path
import json
from typing import Any, Callable, cast

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from IPython.display import display

PROJECT_ROOT = Path('.')
EDA_ARTIFACT_PATH = PROJECT_ROOT / 'artifacts' / 'eda_training_artifacts.pkl'
RULES_PATH = PROJECT_ROOT / 'rules' / 'decision_rules.txt'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'final'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not EDA_ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {
            EDA_ARTIFACT_PATH}. Run eda.ipynb first to materialize the shared training artifacts."
    )

In [111]:
artifacts = pd.read_pickle(EDA_ARTIFACT_PATH)
snapshot_features = artifacts['snapshot_features'].copy()
snapshot_labels = artifacts['snapshot_labels'].copy()
model_frames = artifacts['model_frames']

snapshots = snapshot_features.merge(
    snapshot_labels,
    on=['idSSO', 'reference_date'],
    how='inner',
)
snapshots['reference_date'] = pd.to_datetime(snapshots['reference_date'])
snapshots['reference_month'] = snapshots['reference_date'].dt.strftime('%Y-%m')
snapshots['points_gap_proxy'] = (
    snapshots['reward_threshold_points'] - snapshots['totalPoints'].fillna(0)
).clip(lower=0)

churn_frame = model_frames['churn'].copy()
churn_frame['reference_date'] = pd.to_datetime(churn_frame['reference_date'])
churn_frame['reference_month'] = churn_frame['reference_date'].dt.strftime(
    '%Y-%m')
churn_frame['points_gap_proxy'] = (
    churn_frame['reward_threshold_points'] -
    churn_frame['totalPoints'].fillna(0)
).clip(lower=0)
churn_frame['churn_30_to_60'] = churn_frame['churn_30_to_60'].astype(int)

engagement_frame = model_frames['engagement'].copy()
engagement_frame['reference_date'] = pd.to_datetime(
    engagement_frame['reference_date'])
engagement_frame['reference_month'] = engagement_frame['reference_date'].dt.strftime(
    '%Y-%m')
engagement_frame['points_gap_proxy'] = (
    engagement_frame['reward_threshold_points'] -
    engagement_frame['totalPoints'].fillna(0)
).clip(lower=0)
engagement_frame['re_engage_30d'] = engagement_frame['re_engage_30d'].astype(
    int)

shared_features = artifacts['feature_columns'] + \
    ['points_gap_proxy', 'reference_month']
categorical_features = ['child_age_bucket', 'platform', 'reference_month']

print('Churn rows:', len(churn_frame))
print('Re-engagement rows:', len(engagement_frame))
print('Reference dates:', snapshots['reference_date'].min(
).date(), '->', snapshots['reference_date'].max().date())

Churn rows: 5787
Re-engagement rows: 64924
Reference dates: 2024-04-01 -> 2025-08-01


## Modeling Approach

Both propensities use the same simple stack: shared snapshot features, a temporal holdout on the most recent four reference months, and a class-weighted logistic regression with basic preprocessing.

A temporal holdout means validation is done by time, not by a random row split. For each propensity, the notebook sorts the available `reference_date` snapshots and keeps the most recent four months as the validation set, while all earlier months are used for training. That is closer to the real business setting, because the model is always asked to score future periods using patterns learned from the past.

Validation is done by fitting the model on the earlier reference months, scoring the held-out months, and then measuring discrimination and calibration on that unseen period. The notebook reports ROC AUC, average precision, Brier score, log loss, top-decile precision, and a decile calibration table so we can see both ranking quality and how well the probabilities line up with what actually happened.

The main model is intentionally simple and interpretable. It starts from the shared snapshot feature set produced by `eda.ipynb`, which includes 41 extracted base features, then adds `points_gap_proxy` and the categorical `reference_month` for a total of 43 candidate features. In the current data, all 43 are available and get fed into training for both propensities.

From there, the pipeline imputes missing numeric values with the median, imputes missing categorical values as `missing`, standardizes numeric features, one-hot encodes categorical features, and then fits a `LogisticRegression` model with `class_weight='balanced'`. Later on, the notebook also checks a random forest on the same setup so we can see whether the extra flexibility is worth the drop in interpretability. After validation, the chosen scoring pipeline is retrained on the full available sample for each propensity to generate the exported final scores and CRM decision outputs.


In [112]:
def make_deciles(probabilities: pd.Series) -> pd.Series:
    ranked = probabilities.rank(method='first')
    deciles = pd.qcut(ranked, 10, labels=range(1, 11), duplicates='drop')
    return pd.Series(deciles, index=probabilities.index, dtype='Int64')


def split_by_time(frame: pd.DataFrame, holdout_months: int = 4) -> tuple[pd.DataFrame, pd.DataFrame]:
    reference_dates = sorted(frame['reference_date'].dropna().unique())
    if len(reference_dates) <= holdout_months:
        raise ValueError(
            'Not enough reference dates for the requested holdout split.')
    valid_dates = set(reference_dates[-holdout_months:])
    train_df = cast(
        pd.DataFrame, frame.loc[~frame['reference_date'].isin(valid_dates)].copy())
    valid_df = cast(
        pd.DataFrame, frame.loc[frame['reference_date'].isin(valid_dates)].copy())
    return cast(tuple[pd.DataFrame, pd.DataFrame], (train_df, valid_df))


def build_preprocess(
    feature_columns: list[str],
    categorical_columns: list[str],
    *,
    scale_numeric: bool,
) -> ColumnTransformer:
    usable_categoricals = [
        col for col in categorical_columns if col in feature_columns]
    numeric_columns = [
        col for col in feature_columns if col not in usable_categoricals]
    numeric_steps: list[tuple[str, BaseEstimator]] = [
        ('imputer', SimpleImputer(strategy='median')),
    ]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer(
        transformers=[
            (
                'numeric',
                Pipeline(steps=numeric_steps),
                numeric_columns,
            ),
            (
                'categorical',
                Pipeline(steps=[('imputer', SimpleImputer(
                    strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]),
                usable_categoricals,
            ),
        ]
    )


def build_logistic_pipeline(feature_columns: list[str], categorical_columns: list[str]) -> Pipeline:
    preprocess = build_preprocess(
        feature_columns, categorical_columns, scale_numeric=True)
    return Pipeline(
        steps=[
            ('preprocess', preprocess),
            ('model', LogisticRegression(class_weight='balanced',
             max_iter=1000, random_state=42, solver='liblinear')),
        ]
    )


def build_random_forest_pipeline(feature_columns: list[str], categorical_columns: list[str]) -> Pipeline:
    preprocess = build_preprocess(
        feature_columns, categorical_columns, scale_numeric=False)
    return Pipeline(
        steps=[
            ('preprocess', preprocess),
            ('model', RandomForestClassifier(
                n_estimators=400,
                max_depth=10,
                min_samples_leaf=20,
                class_weight='balanced_subsample',
                n_jobs=-1,
                random_state=42,
            )),
        ]
    )


def clean_feature_name(feature_name: str) -> str:
    return feature_name.replace('numeric__', '').replace('categorical__', '')


def extract_feature_relevance(model_pipeline: Pipeline, *, top_n: int = 15) -> pd.DataFrame:
    feature_names = [
        clean_feature_name(name)
        for name in model_pipeline.named_steps['preprocess'].get_feature_names_out()
    ]
    model_step = model_pipeline.named_steps['model']
    if hasattr(model_step, 'coef_'):
        feature_relevance = pd.DataFrame(
            {'feature': feature_names, 'relevance': model_step.coef_[0]}
        )
        feature_relevance['abs_relevance'] = feature_relevance['relevance'].abs()
        feature_relevance['direction'] = np.where(
            feature_relevance['relevance'] >= 0,
            'raises score',
            'lowers score',
        )
        feature_relevance['relevance_type'] = 'coefficient'
    elif hasattr(model_step, 'feature_importances_'):
        feature_relevance = pd.DataFrame(
            {'feature': feature_names, 'relevance': model_step.feature_importances_}
        )
        feature_relevance['abs_relevance'] = feature_relevance['relevance']
        feature_relevance['direction'] = 'non-directional'
        feature_relevance['relevance_type'] = 'feature_importance'
    else:
        return pd.DataFrame(
            columns=['feature', 'relevance', 'abs_relevance',
                     'direction', 'relevance_type']
        )
    return feature_relevance.sort_values('abs_relevance', ascending=False).head(top_n)


def calibration_table(frame: pd.DataFrame, probability_column: str, label_column: str) -> pd.DataFrame:
    scored = frame.copy()
    scored['score_decile'] = make_deciles(scored[probability_column])
    return (
        scored.groupby('score_decile', dropna=False)
        .agg(rows=(label_column, 'size'), avg_prediction=(probability_column, 'mean'), actual_rate=(label_column, 'mean'))
        .reset_index()
        .sort_values('score_decile')
    )


def evaluate_predictions(frame: pd.DataFrame, probability_column: str, label_column: str) -> pd.Series:
    y_true = frame[label_column].astype(int)
    y_prob = frame[probability_column].clip(1e-6, 1 - 1e-6)
    top_n = max(1, int(np.ceil(len(frame) * 0.10)))
    top_rate = float(frame.nlargest(top_n, probability_column)
                     [label_column].mean())
    return pd.Series({
        'rows': int(len(frame)),
        'positive_rate': float(y_true.mean()),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'average_precision': float(average_precision_score(y_true, y_prob)),
        'brier_score': float(brier_score_loss(y_true, y_prob)),
        'log_loss': float(log_loss(y_true, y_prob)),
        'top_decile_precision': top_rate,
    })


def fit_propensity_model(
    frame: pd.DataFrame,
    *,
    label_column: str,
    score_column: str,
    feature_columns: list[str],
    categorical_columns: list[str],
    pipeline_builder: Callable[[list[str], list[str]],
                               Pipeline] = build_logistic_pipeline,
    model_family: str = 'logistic_regression',
) -> dict[str, Any]:
    usable_features = [
        col for col in feature_columns if col in frame.columns and not cast(pd.Series, frame[col]).dropna().empty]
    model_frame = frame.copy()
    for column in usable_features:
        if column in categorical_columns:
            model_frame[column] = model_frame[column].astype(
                'string').fillna('missing').astype(str)
        else:
            model_frame[column] = pd.to_numeric(
                model_frame[column], errors='coerce').astype(float)

    train_df, valid_df = split_by_time(model_frame)
    validation_model = pipeline_builder(usable_features, categorical_columns)
    validation_model.fit(train_df[usable_features],
                         train_df[label_column].astype(int))

    valid_scored = valid_df.copy()
    valid_scored[score_column] = validation_model.predict_proba(
        valid_df[usable_features])[:, 1]

    final_model = pipeline_builder(usable_features, categorical_columns)
    final_model.fit(model_frame[usable_features],
                    model_frame[label_column].astype(int))

    full_scored = model_frame.copy()
    full_scored[score_column] = final_model.predict_proba(
        model_frame[usable_features])[:, 1]

    feature_relevance = extract_feature_relevance(final_model)
    top_coefficients = pd.DataFrame(
        columns=['feature', 'coefficient', 'abs_coefficient'])
    if not feature_relevance.empty and feature_relevance['relevance_type'].iloc[0] == 'coefficient':
        top_coefficients = feature_relevance.rename(
            columns={'relevance': 'coefficient',
                     'abs_relevance': 'abs_coefficient'}
        )[['feature', 'coefficient', 'abs_coefficient']]

    return {
        'validation_model': validation_model,
        'final_model': final_model,
        'features': usable_features,
        'validation_scores': valid_scored,
        'full_scores': full_scored,
        'validation_metrics': evaluate_predictions(valid_scored, score_column, label_column),
        'calibration': calibration_table(valid_scored, score_column, label_column),
        'feature_relevance': feature_relevance,
        'top_coefficients': top_coefficients,
        'model_family': model_family,
    }

## Validation Metrics Output

The table below summarizes model performance on the temporal validation holdout. Each row is one propensity model, and each column describes either the size of the validation sample, the base rate of the target, or how well the model ranked and calibrated users in the held-out months.

For the re-engagement row, the evaluation audience is the EDA-defined historical-activity population. The positive class is `re_engage_30d`, meaning at least one future `scan`, `mission`, or `redeem` event within 30 days.

- `model`: the propensity being evaluated (`churn` or `re_engage`)
- `rows`: number of validation snapshots scored for that model
- `positive_rate`: share of validation rows where the target outcome actually happened
- `roc_auc`: ranking quality across positives vs negatives; higher is better, with `0.5` close to random
- `average_precision`: precision-recall summary, especially useful for imbalanced targets; higher is better
- `brier_score`: average squared error of predicted probabilities; lower is better
- `log_loss`: probability error that penalizes confident wrong predictions more heavily; lower is better
- `top_decile_precision`: realized positive rate among the top 10% highest-scored users; higher is better and useful for campaign targeting


In [113]:
churn_results: dict[str, Any] = fit_propensity_model(
    churn_frame,
    label_column='churn_30_to_60',
    score_column='churn_30_to_60_prob',
    feature_columns=shared_features,
    categorical_columns=categorical_features,
)

engagement_results: dict[str, Any] = fit_propensity_model(
    engagement_frame,
    label_column='re_engage_30d',
    score_column='re_engage_30d_prob',
    feature_columns=shared_features,
    categorical_columns=categorical_features,
)

validation_metrics = pd.DataFrame([
    {'model': 'churn', **churn_results['validation_metrics'].to_dict()},
    {'model': 're_engage', **
        engagement_results['validation_metrics'].to_dict()},
]).round(4)

validation_metrics

,model,rows,positive_rate,roc_auc,average_precision,brier_score,log_loss,top_decile_precision
0,churn,1251.0,0.6563,0.7981,0.8763,0.2188,0.6288,0.9524
1,re_engage,19115.0,0.3813,0.9635,0.9229,0.0709,0.2382,0.9582


## Model Family Check

The next table compares the main logistic regression baseline with a random forest trained on the same features, audience definitions, and temporal holdout. This is a model-family check, not a production switch by itself: the goal is to see whether a more flexible model materially improves ranking quality before changing the main scoring setup.

For this project, logistic regression is still the main family used for the exported score files, the backend, and the CRM engine. The random forest is kept here as a benchmark because it scores better, but it is less direct to explain and less convenient to map back into business rules.

In [114]:
rf_churn_results: dict[str, Any] = fit_propensity_model(
    churn_frame,
    label_column='churn_30_to_60',
    score_column='churn_30_to_60_rf_prob',
    feature_columns=shared_features,
    categorical_columns=categorical_features,
    pipeline_builder=build_random_forest_pipeline,
    model_family='random_forest',
)

rf_engagement_results: dict[str, Any] = fit_propensity_model(
    engagement_frame,
    label_column='re_engage_30d',
    score_column='re_engage_30d_rf_prob',
    feature_columns=shared_features,
    categorical_columns=categorical_features,
    pipeline_builder=build_random_forest_pipeline,
    model_family='random_forest',
)

model_family_comparison = pd.DataFrame([
    {'target': 'churn', 'model_family': 'logistic_regression',
        **churn_results['validation_metrics'].to_dict()},
    {'target': 'churn', 'model_family': 'random_forest',
        **rf_churn_results['validation_metrics'].to_dict()},
    {'target': 're_engage', 'model_family': 'logistic_regression',
        **engagement_results['validation_metrics'].to_dict()},
    {'target': 're_engage', 'model_family': 'random_forest',
        **rf_engagement_results['validation_metrics'].to_dict()},
]).round(4)

model_family_comparison

,target,model_family,rows,positive_rate,roc_auc,average_precision,brier_score,log_loss,top_decile_precision
0,churn,logistic_regression,1251.0,0.6563,0.7981,0.8763,0.2188,0.6288,0.9524
1,churn,random_forest,1251.0,0.6563,0.8062,0.8804,0.2096,0.6042,0.9603
2,re_engage,logistic_regression,19115.0,0.3813,0.9635,0.9229,0.0709,0.2382,0.9582
3,re_engage,random_forest,19115.0,0.3813,0.9690,0.9395,0.0679,0.2215,0.9759


## Feature Relevance Check

Scores are only part of the story, so the next tables show which features each model leans on most. The relevance is calculated differently for the two model families.

For logistic regression, relevance is the fitted coefficient after preprocessing. A positive coefficient means that feature pushes the score up, and a negative coefficient means it pushes the score down. The table is sorted by absolute coefficient size, so the biggest effects show up first.

For the random forest, relevance is the built-in `feature_importances_` value from the fitted model. Those numbers are non-directional: they show how much each feature helped the tree ensemble split the data, but not whether the feature increases or decreases the final score.

In plain terms: the random forest is stronger on the holdout, but the logistic model is still easier to explain to business users and easier to turn into simple rules. That tradeoff matters here, so the backend and CRM engine still use the logistic-regression path as the default scoring setup.


In [115]:
def format_feature_relevance(result: dict[str, Any], target: str) -> pd.DataFrame:
    relevance = result['feature_relevance'].copy()
    relevance.insert(0, 'target', target)
    relevance.insert(1, 'model_family', result['model_family'])
    return relevance[[
        'target',
        'model_family',
        'feature',
        'relevance_type',
        'direction',
        'relevance',
    ]].round(4)


churn_feature_relevance = pd.concat([
    format_feature_relevance(churn_results, 'churn'),
    format_feature_relevance(rf_churn_results, 'churn'),
], ignore_index=True)

engagement_feature_relevance = pd.concat([
    format_feature_relevance(engagement_results, 're_engage'),
    format_feature_relevance(rf_engagement_results, 're_engage'),
], ignore_index=True)

print('Churn feature relevance')
display(churn_feature_relevance)
print('Re-engagement feature relevance')
display(engagement_feature_relevance)

Churn feature relevance


,target,model_family,feature,relevance_type,direction,relevance
0,churn,logistic_regression,platform_missing,coefficient,raises score,1.6318
1,churn,logistic_regression,points_earned_lifetime_proxy,coefficient,lowers score,-1.0794
2,churn,logistic_regression,reference_month_2025-06,coefficient,raises score,0.6463
3,churn,logistic_regression,reference_month_2025-01,coefficient,lowers score,-0.6246
4,churn,logistic_regression,points_redeemed_lifetime,coefficient,raises score,0.5692
5,churn,logistic_regression,platform_android,coefficient,lowers score,-0.5675
6,churn,logistic_regression,platform_ios,coefficient,lowers score,-0.5146
7,churn,logistic_regression,totalPoints,coefficient,raises score,0.4997
8,churn,logistic_regression,points_gap_proxy,coefficient,raises score,0.3354
9,churn,logistic_regression,reference_month_2024-07,coefficient,raises score,0.3347


Re-engagement feature relevance


,target,model_family,feature,relevance_type,direction,relevance
0,re_engage,logistic_regression,days_since_last_activity,coefficient,lowers score,-4.9240
1,re_engage,logistic_regression,days_since_last_login,coefficient,raises score,3.1289
2,re_engage,logistic_regression,points_earned_lifetime_proxy,coefficient,raises score,1.6240
3,re_engage,logistic_regression,points_redeemed_lifetime,coefficient,lowers score,-1.2039
4,re_engage,logistic_regression,platform_missing,coefficient,lowers score,-1.1342
5,re_engage,logistic_regression,reference_month_2024-10,coefficient,lowers score,-1.1305
6,re_engage,logistic_regression,mission_count_60d,coefficient,raises score,0.7966
7,re_engage,logistic_regression,totalPoints,coefficient,lowers score,-0.6398
8,re_engage,logistic_regression,missions_completed_60d,coefficient,lowers score,-0.5685
9,re_engage,logistic_regression,child_age_bucket_pregnancy,coefficient,lowers score,-0.4796


## Churn and Re-engagement Calibration

The first table checks whether the churn probabilities line up with what actually happened in the holdout. The rows are score deciles from low risk to high risk.

The second table does the same check for `re_engage_30d`. It helps show whether a score bucket like `0.60` or `0.80` means roughly what it says on the holdout.


In [116]:
print('Churn calibration')
display(churn_results['calibration'])

print('Re-engagement calibration')
display(engagement_results['calibration'])

Churn calibration


,score_decile,rows,avg_prediction,actual_rate
0,1,126,0.131421,0.285714
1,2,125,0.202870,0.336000
2,3,125,0.253262,0.408000
3,4,125,0.308315,0.536000
4,5,125,0.366578,0.696000
5,6,125,0.434623,0.736000
6,7,125,0.523822,0.800000
7,8,125,0.654059,0.944000
8,9,125,0.758254,0.872000
9,10,125,0.863484,0.952000


Re-engagement calibration


,score_decile,rows,avg_prediction,actual_rate
0,1,1912,0.000109,0.000000
1,2,1911,0.000650,0.003663
2,3,1912,0.004336,0.003661
3,4,1911,0.021294,0.010989
4,5,1912,0.105543,0.053870
5,6,1911,0.420628,0.294610
6,7,1911,0.718002,0.692831
7,8,1912,0.852318,0.862448
8,9,1911,0.925610,0.933019
9,10,1912,0.973679,0.958159


## Rule Design

The CRM engine keeps the model score as the main signal and only adds a small set of support variables that held up in the holdout. The point is to keep the rules explainable and operational, not to rebuild the model by hand.

For churn, the rules only add `is_near_graduation` and channel eligibility on top of `churn_30_to_60_prob`. For re-engagement, the rules only add `days_since_last_activity`, `has_ever_redeemed`, `scan_count_60d`, and channel eligibility on top of `re_engage_30d_prob`.

The score thresholds are explicit on purpose. In the holdout, `churn_30_to_60_prob >= 0.75` captured the clearest high-risk churn pocket, while `re_engage_30d_prob >= 0.60` worked as a hot-user threshold and `re_engage_30d_prob >= 0.40` worked as a warmer, lower-priority threshold.


In [117]:
def load_rules(path: Path) -> list[dict[str, str]]:
    rules: list[dict[str, str]] = []
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        parts = {}
        for part in line.split('|'):
            key, value = part.split('=', 1)
            parts[key] = value
        rules.append(parts)
    return rules


def apply_rules(scored_frame: pd.DataFrame, audience: str, rules: list[dict[str, str]]) -> pd.DataFrame:
    segment_field = 'churn_type' if audience == 'churn' else 'engagement_segment'
    action_field = 'recommended_campaign' if audience == 'churn' else 'recommended_action'
    decisions = pd.DataFrame(index=scored_frame.index)
    decisions['matched_rule'] = pd.NA
    decisions[segment_field] = pd.NA
    decisions[action_field] = pd.NA
    decisions['recommended_channel'] = pd.NA
    decisions['priority'] = pd.NA
    remaining = pd.Series(True, index=scored_frame.index)
    for rule in [item for item in rules if item['audience'] == audience]:
        if rule['condition'] == 'True':
            rule_mask = remaining.copy()
        else:
            rule_mask = scored_frame.eval(
                rule['condition'], engine='python').fillna(False).astype(bool)
            rule_mask &= remaining
        decisions.loc[rule_mask, 'matched_rule'] = rule['rule_name']
        decisions.loc[rule_mask, segment_field] = rule['segment']
        decisions.loc[rule_mask, action_field] = rule['action']
        decisions.loc[rule_mask,
                      'recommended_channel'] = rule['recommended_channel']
        decisions.loc[rule_mask, 'priority'] = rule['priority']
        remaining &= ~rule_mask
    base_columns = ['idSSO', 'reference_date']
    if audience == 'churn':
        base_columns += ['churn_30_to_60_prob', 'risk_decile']
    else:
        base_columns += ['re_engage_30d_prob']
    return pd.concat([scored_frame[base_columns], decisions], axis=1)


rules = load_rules(RULES_PATH)

churn_scores = churn_results['full_scores'][[
    'idSSO', 'reference_date', 'churn_30_to_60_prob']].copy()
churn_scores['risk_decile'] = make_deciles(churn_scores['churn_30_to_60_prob'])
churn_engine_input = churn_results['full_scores'].copy()
churn_engine_input['risk_decile'] = make_deciles(
    churn_engine_input['churn_30_to_60_prob'])
churn_engine = apply_rules(churn_engine_input, 'churn', rules)

engagement_scores = engagement_results['full_scores'][[
    'idSSO', 'reference_date', 're_engage_30d_prob']].copy()
engagement_engine = apply_rules(
    engagement_results['full_scores'], 'engagement', rules)

latest_reference_date = snapshots['reference_date'].max()
current_outputs = {
    'churn_scores_current': churn_scores[churn_scores['reference_date'] == latest_reference_date],
    'engagement_scores_current': engagement_scores[engagement_scores['reference_date'] == latest_reference_date],
    'churn_engine_current': churn_engine[churn_engine['reference_date'] == latest_reference_date],
    'engagement_engine_current': engagement_engine[engagement_engine['reference_date'] == latest_reference_date],
}

for name, frame in {
    'churn_scores_all': churn_scores,
    'engagement_scores_all': engagement_scores,
    'churn_engine_all': churn_engine,
    'engagement_engine_all': engagement_engine,
    **current_outputs,
}.items():
    frame.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)

validation_metrics.to_csv(OUTPUT_DIR / 'validation_metrics.csv', index=False)
churn_results['calibration'].to_csv(
    OUTPUT_DIR / 'churn_calibration.csv', index=False)
engagement_results['calibration'].to_csv(
    OUTPUT_DIR / 'engagement_calibration.csv', index=False)
churn_results['top_coefficients'].to_csv(
    OUTPUT_DIR / 'churn_top_coefficients.csv', index=False)
engagement_results['top_coefficients'].to_csv(
    OUTPUT_DIR / 'engagement_top_coefficients.csv', index=False)

with (OUTPUT_DIR / 'run_summary.json').open('w') as handle:
    json.dump({
        'latest_reference_date': str(latest_reference_date.date()),
        'rules_path': str(RULES_PATH),
        'eda_artifact_path': str(EDA_ARTIFACT_PATH),
        'churn_rows': int(len(churn_scores)),
        'engagement_rows': int(len(engagement_scores)),
    }, handle, indent=2)

print('Churn engine sample')
display(churn_engine.head(10))

print('Re-engagement engine sample')
display(engagement_engine.head(10))

Churn engine sample


,idSSO,reference_date,churn_30_to_60_prob,risk_decile,matched_rule,churn_type,recommended_campaign,recommended_channel,priority
25,095278ec491f259c3430bed05bf39a907bde85d396c49b...,2024-04-01,0.747856,8,watchlist_email,watchlist_churn,soft_save_nudge,email,medium
40,e71c5a177b0c96d6be1448eb21105dd09ea6d9f18a5198...,2024-04-01,0.973697,10,preventable_push,preventable_churn,preventable_churn_core,push,high
47,9449088472699f3887acee3b5ce296472ea1a89331ad19...,2024-04-01,0.949918,10,preventable_push,preventable_churn,preventable_churn_core,push,high
64,a387de25c621a0bdbe823995be6402b9322c2b37524a58...,2024-04-01,0.268523,3,default_churn,monitor_only,no_action,none,low
83,0eb7a27640f6d28409bd9a4fb43700491893a36da5b0c3...,2024-04-01,0.100212,1,default_churn,monitor_only,no_action,none,low
101,56b1bd933953fab9e44690cb07362770653114ce6da362...,2024-04-01,0.715339,8,watchlist_email,watchlist_churn,soft_save_nudge,email,medium
115,16d56e836858281fc7434dda7d354722909b1bc12a5dd6...,2024-04-01,0.617837,6,watchlist_email,watchlist_churn,soft_save_nudge,email,medium
120,0a43f40eaaab5f42e03dedf03a8f792d5c4f7db30da248...,2024-04-01,0.187492,2,default_churn,monitor_only,no_action,none,low
142,0a9c6bf154af70279bca1ce241198c715319b660486699...,2024-04-01,0.188389,2,default_churn,monitor_only,no_action,none,low
144,a1aa15552c24fd215e968fd8993873eace14b8c6b74f52...,2024-04-01,0.625175,6,default_churn,monitor_only,no_action,none,low


Re-engagement engine sample


,idSSO,reference_date,re_engage_30d_prob,matched_rule,engagement_segment,recommended_action,recommended_channel,priority
0,4f12b21d8e9c49d1a4085f21f2d9d5ec708791b341ca25...,2024-04-01,0.538166,default_engagement,watchlist,light_touch_nurture,email,low
1,acf181fcd82a7451e37f3c33ac7008cb1906d4f0167717...,2024-04-01,0.735530,hot_user_push,hot_user,mission_bundle_nudge,push,high
6,13138414e028eff18f0b705cebe5cb5c28bcd645358efd...,2024-04-01,0.858142,hot_user_push,hot_user,mission_bundle_nudge,push,high
8,e29ef81dda850778b56930bbea87228a84fb58abde2a13...,2024-04-01,0.967111,not_eligible,not_campaign_eligible,no_action,none,low
10,0b6e422b05ce5daa19732c9ff2a5923058e87e26c88fe2...,2024-04-01,0.271453,default_engagement,watchlist,light_touch_nurture,email,low
13,65720589fa0e5fe8d96a518c51df0e450249f81f3380d9...,2024-04-01,0.273080,default_engagement,watchlist,light_touch_nurture,email,low
19,6260b7930804e6d735f5a9bedf6dec42725544c7fa6443...,2024-04-01,0.994865,hot_user_push,hot_user,mission_bundle_nudge,push,high
24,50eae48017f6e9726c021e17df3c607e7cbe51d05ab9ee...,2024-04-01,0.326577,default_engagement,watchlist,light_touch_nurture,email,low
25,095278ec491f259c3430bed05bf39a907bde85d396c49b...,2024-04-01,0.146745,default_engagement,watchlist,light_touch_nurture,email,low
26,1f33ef18f2c977721c5cca94de37daf11cb87df640eff8...,2024-04-01,0.962793,hot_user_push,hot_user,mission_bundle_nudge,push,high
